In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

SEED = 42
DATA_PATH = Path("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
TASKS = ["direction", "value"]
DEFAULT_LOOKBACK_DAYS = 365
DEFAULT_EXPERIMENTS = [
    {"hidden_size": hs, "lr": lr, "batch_size": bs, "weight_decay": wd}
    for hs in [32, 64, 128]
    for lr in [1e-3, 3e-4, 1e-4]
    for bs in [16, 32]
    for wd in [1e-5, 1e-4]
]

MAX_EPOCHS = 100
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Configured Tasks: {TASKS}")
print(f"Total experiments per target: {len(DEFAULT_EXPERIMENTS)}")

Using device: cuda
Configured Tasks: ['direction', 'value']
Total experiments per target: 36


# Shallow LSTM benchmark

This notebook builds a single-layer LSTM for both a binary next-day direction task and a regression task on year-over-year value change.
The setup keeps the evaluation chronological, fits normalization only on training data, and reports both overfitting signals and temporal drift on the held-out period.

## Data Processing

In [3]:
def load_and_prepare_yoy_data(
    data_path: Path, lookback_days: int = 365
) -> tuple[pd.DataFrame, list[str]]:
    """
    Load combined daily + targets data, build year-over-year sequences.
    Identifies year-end dates (where targets are not NaN) and extracts lookback_days
    of daily features ending at each year-end, then creates YoY direction and value labels.
    """
    df = pd.read_csv(data_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()
    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            # Prior year record for comparison
            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]

            # Extract window of daily data
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[
                (company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)
            ].copy()

            if len(window) < 200:
                continue

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append(
                {
                    "company": company,
                    "year": year,
                    "year_end_date": year_end_date,
                    "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                    "current_ebitda": year_end_row["EBITDA"],
                    "current_net_income": year_end_row["Net_Income"],
                    "current_roa": year_end_row["ROA"],
                    "prior_ebitda": prior_row["EBITDA"],
                    "prior_net_income": prior_row["Net_Income"],
                    "prior_roa": prior_row["ROA"],
                }
            )

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]

            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_direction = 1 if current_val > prior_val else 0
            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)  # Normalized change

            data_records.append(
                {
                    "company": seq["company"],
                    "year": seq["year"],
                    "year_end_date": seq["year_end_date"],
                    "target": target_name,
                    "label_direction": label_direction,
                    "label_value": label_value,
                    "window_data": seq["window_data"],
                }
            )

    return pd.DataFrame(data_records), feature_cols

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

print(f"Total sequences collected: {len(data_df):,}")
print(f"Features per sequence: {len(feature_cols)}")
print(f"Date range: {data_df['year_end_date'].min().date()} to {data_df['year_end_date'].max().date()}" if len(data_df) > 0 else "No data")
print(f"Companies: {data_df['company'].nunique()}" if len(data_df) > 0 else "No data")
print(f"\nBreakdown by target:")
for target in PREDICTION_TARGETS:
    target_data = data_df[data_df["target"] == target]
    if len(target_data) > 0:
        pos_rate = target_data["label_direction"].mean()
        print(f"  {target}: {len(target_data)} samples, positive rate: {pos_rate:.4f}")
    else:
        print(f"  {target}: 0 samples")

Total sequences collected: 5,826
Features per sequence: 25
Date range: 2010-12-30 to 2025-12-30
Companies: 169

Breakdown by target:
  EBITDA: 1909 samples, positive rate: 0.6401
  Net_Income: 1959 samples, positive rate: 0.5937
  ROA: 1958 samples, positive rate: 0.5209


In [4]:
class YoYSequenceDataset(Dataset):
    """Dataset for year-over-year fundamental prediction."""

    def __init__(self, data_df: pd.DataFrame, max_seq_length: int, task: str, scaler: StandardScaler = None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.task = task
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()  # shape: (seq_len, features)

        if self.scaler is not None:
            window = self.scaler.transform(window)

        # Pad to max_seq_length (PRE-PADDING: add zeros at beginning so LSTM processes real data last)
        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])

        label_col = f"label_{self.task}"
        target = np.float32(row[label_col])
        return torch.from_numpy(window), torch.tensor(target)


def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split data chronologically by year."""
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

In [5]:
train_data, val_data, test_data = split_by_year(data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

print(f"\nChronological split (train ≤ {TRAIN_YEAR_CUTOFF}, val ≤ {VALID_YEAR_CUTOFF}, test > {VALID_YEAR_CUTOFF}):")
print(f"  Train: {len(train_data)} samples ({train_data['year'].min():.0f}–{train_data['year'].max():.0f})")
print(f"  Val: {len(val_data)} samples ({val_data['year'].min():.0f}–{val_data['year'].max():.0f})")
print(f"  Test: {len(test_data)} samples ({test_data['year'].min():.0f}–{test_data['year'].max():.0f})")

scaler = StandardScaler()
train_windows = np.vstack([row for row in train_data["window_data"]])
scaler.fit(train_windows)
print(f"\nScaler fitted on {len(train_windows)} training windows")

max_seq_length = max(len(row) for row in data_df["window_data"])
print(f"Max sequence length: {max_seq_length}")


Chronological split (train ≤ 2019, val ≤ 2021, test > 2021):
  Train: 3471 samples (2010–2019)
  Val: 899 samples (2020–2021)
  Test: 1456 samples (2022–2025)

Scaler fitted on 873126 training windows
Max sequence length: 260


In [6]:
class ShallowLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, task: str = "direction"):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, 1)
        self.task = task

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        hidden = self.dropout(lstm_out[:, -1, :])
        logits = self.head(hidden).squeeze(-1)
        return logits


@torch.no_grad()
def collect_predictions(
    model: nn.Module, loader: DataLoader, criterion: nn.Module, task: str = "direction"
) -> tuple[dict, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds: list[np.ndarray] = []
    all_targets: list[np.ndarray] = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

        if task == "direction":
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.cpu().numpy())

        all_targets.append(batch_y.cpu().numpy())

    if total_count == 0:
        empty_metrics = {"loss": np.nan, "accuracy": np.nan, "roc_auc": np.nan}
        return empty_metrics, np.array([]), np.array([])

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    if task == "direction":
        y_pred_binary = (y_pred >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_true.astype(int), y_pred)
        except ValueError:
            auc = np.nan
        metrics = {
            "loss": total_loss / total_count,
            "accuracy": accuracy_score(y_true.astype(int), y_pred_binary),
            "roc_auc": auc,
        }
        return metrics, y_true.astype(int), y_pred
    else:
        mae = np.mean(np.abs(y_pred - y_true))
        metrics = {
            "loss": total_loss / total_count,
            "mae": mae,
        }
        return metrics, y_true, y_pred


def train_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, task: str = "direction") -> dict:
    model.train()
    total_loss = 0.0
    total_count = 0
    all_preds: list[np.ndarray] = []
    all_targets: list[np.ndarray] = []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

        if task == "direction":
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.detach().cpu().numpy())

        all_targets.append(batch_y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    if task == "direction":
        y_pred_binary = (y_pred >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_true.astype(int), y_pred)
        except ValueError:
            auc = np.nan
        return {
            "loss": total_loss / total_count,
            "accuracy": accuracy_score(y_true.astype(int), y_pred_binary),
            "roc_auc": auc,
        }
    else:
        mae = np.mean(np.abs(y_pred - y_true))
        return {"loss": total_loss / total_count, "mae": mae}


def make_loaders(train_data: pd.DataFrame, val_data: pd.DataFrame, test_data: pd.DataFrame, batch_size: int, max_seq_length: int, task: str):
    train_ds = YoYSequenceDataset(train_data, max_seq_length=max_seq_length, task=task, scaler=scaler)
    val_ds = YoYSequenceDataset(val_data, max_seq_length=max_seq_length, task=task, scaler=scaler)
    test_ds = YoYSequenceDataset(test_data, max_seq_length=max_seq_length, task=task, scaler=scaler)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

    return train_loader, val_loader, test_loader


def fit_single_experiment(target_name: str, config: dict, max_seq_length: int, task: str):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Filter data for this target
    target_train = train_data[train_data["target"] == target_name].copy()
    target_val = val_data[val_data["target"] == target_name].copy()
    target_test = test_data[test_data["target"] == target_name].copy()

    if len(target_train) < 5 or len(target_val) < 2 or len(target_test) < 2:
        return None

    train_loader, val_loader, test_loader = make_loaders(target_train, target_val, target_test, config["batch_size"], max_seq_length, task)

    model = ShallowLSTMClassifier(input_size=len(feature_cols), hidden_size=config["hidden_size"], task=task).to(DEVICE)

    if task == "direction":
        pos_weight = torch.tensor(
            (len(target_train) - target_train["label_direction"].sum()) / max(1, target_train["label_direction"].sum()),
            dtype=torch.float32,
            device=DEVICE,
        )
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.HuberLoss(delta=1.0)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    history_rows = []
    best_state = None
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, task=task)
        val_metrics, _, _ = collect_predictions(model, val_loader, criterion, task=task)
        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "val_loss": val_metrics["loss"],
                **(
                    {"train_acc": train_metrics.get("accuracy"), "val_acc": val_metrics.get("accuracy")}
                    if task == "direction"
                    else {"train_mae": train_metrics.get("mae"), "val_mae": val_metrics.get("mae")}
                ),
            }
        )

        if val_metrics["loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["loss"]
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_metrics, _, _ = collect_predictions(model, train_loader, criterion, task=task)
    val_metrics, _, _ = collect_predictions(model, val_loader, criterion, task=task)
    test_metrics, test_true, test_pred = collect_predictions(model, test_loader, criterion, task=task)

    target_test_frame = target_test.reset_index(drop=True).copy()
    target_test_frame["true"] = test_true
    target_test_frame["pred"] = test_pred if task != "direction" else (test_pred >= 0.5).astype(int)

    return {
        "target": target_name,
        "config": config,
        "model": model,
        "history": pd.DataFrame(history_rows),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_frame": target_test_frame,
        "train_size": len(target_train),
        "val_size": len(target_val),
        "test_size": len(target_test),
    }

In [11]:
def extract_mtl_df(df: pd.DataFrame) -> pd.DataFrame:
    """Group by company and year to build a unified array of targets."""
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        window_data = group.iloc[0]["window_data"]
        year_end_date = group.iloc[0]["year_end_date"]

        label_direction = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)

        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_direction[i] = t_row.iloc[0]["label_direction"]
                label_value[i] = t_row.iloc[0]["label_value"]

        mtl_records.append({
            "company": company,
            "year": year,
            "year_end_date": year_end_date,
            "window_data": window_data,
            "label_direction": label_direction,
            "label_value": label_value
        })
    return pd.DataFrame(mtl_records)

mtl_train_data = extract_mtl_df(train_data)
mtl_val_data = extract_mtl_df(val_data)
mtl_test_data = extract_mtl_df(test_data)

class YoYSequenceDatasetMTL(Dataset):
    def __init__(self, data_df: pd.DataFrame, max_seq_length: int, task: str, scaler: StandardScaler = None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.task = task
        self.scaler = scaler

    def __len__(self) -> int:
        return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()

        if self.scaler is not None:
            window = self.scaler.transform(window)

        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])

        target = row[f"label_{self.task}"]

        # Create a mask where True means we have data for that target
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)

        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool)

class ShallowLSTMClassifierMTL(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), task: str = "direction"):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, num_targets)
        self.task = task

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        hidden = self.dropout(lstm_out[:, -1, :])
        logits = self.head(hidden)
        return logits

def train_epoch_mtl(model, loader, optimizer, task="direction"):
    model.train()
    total_loss = 0.0
    total_count = 0

    if task == "direction":
        criterion = nn.BCEWithLogitsLoss(reduction='none')
    else:
        criterion = nn.HuberLoss(delta=1.0, reduction='none')

    for batch_x, batch_y, batch_mask in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        batch_mask = batch_mask.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x)

        loss_matrix = criterion(logits, batch_y)
        masked_loss = loss_matrix[batch_mask]

        if masked_loss.numel() > 0:
            loss = masked_loss.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += float(loss.item()) * batch_x.size(0)
            total_count += batch_x.size(0)

    return {"loss": total_loss / max(1, total_count)}

@torch.no_grad()
def collect_predictions_mtl(model, loader, task="direction"):
    model.eval()
    total_loss = 0.0
    total_count = 0

    all_preds = []
    all_targets = []
    all_masks = []

    if task == "direction":
        criterion = nn.BCEWithLogitsLoss(reduction='none')
    else:
        criterion = nn.HuberLoss(delta=1.0, reduction='none')

    for batch_x, batch_y, batch_mask in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        batch_mask = batch_mask.to(DEVICE)

        logits = model(batch_x)
        loss_matrix = criterion(logits, batch_y)
        masked_loss = loss_matrix[batch_mask]

        if masked_loss.numel() > 0:
            total_loss += float(masked_loss.mean().item()) * batch_x.size(0)
            total_count += batch_x.size(0)

        if task == "direction":
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.cpu().numpy())

        all_targets.append(batch_y.cpu().numpy())
        all_masks.append(batch_mask.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks)

    metrics = {"loss": total_loss / max(1, total_count)}

    # Calculate metrics for each target individually
    target_metrics = {}
    for i, target_name in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue

        target_true = y_true[m, i]
        target_pred = y_pred[m, i]

        if task == "direction":
            target_pred_binary = (target_pred >= 0.5).astype(int)
            acc = accuracy_score(target_true.astype(int), target_pred_binary)
            target_metrics[f"{target_name}_acc"] = acc
        else:
            mae = np.mean(np.abs(target_pred - target_true))
            target_metrics[f"{target_name}_mae"] = mae

    metrics.update(target_metrics)
    return metrics, y_true, y_pred, mask

def fit_single_experiment_mtl(config: dict, max_seq_length: int, task: str):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, task, scaler)
    val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, task, scaler)
    test_ds = YoYSequenceDatasetMTL(mtl_test_data, max_seq_length, task, scaler)

    if len(mtl_train_data) < 5 or len(mtl_val_data) < 2 or len(mtl_test_data) < 2:
        return None

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, drop_last=False)

    model = ShallowLSTMClassifierMTL(input_size=len(feature_cols), hidden_size=config["hidden_size"], task=task).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_state = None
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_epoch_mtl(model, train_loader, optimizer, task)
        val_metrics, _, _, _ = collect_predictions_mtl(model, val_loader, task)

        if val_metrics["loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["loss"]
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_metrics, _, _, _ = collect_predictions_mtl(model, train_loader, task)
    val_metrics, _, _, _ = collect_predictions_mtl(model, val_loader, task)
    test_metrics, test_true, test_pred, test_mask = collect_predictions_mtl(model, test_loader, task)

    return {
        "config": config,
        "model": model,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

## Multi-Task Learning (MTL) Implementations

In [ ]:
run_results_mtl_direction = []

print(f"\n{'#'*90}")
print("### MTL: DIRECTION")
print(f"{'#'*90}")

for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
    print(
        f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
    )
    result = fit_single_experiment_mtl(
        config=config,
        max_seq_length=max_seq_length,
        task="direction",
    )

    if result is not None:
        run_results_mtl_direction.append(result)
        metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
        metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}\n"
        metrics_str += "Test Accuracies: "
        for target in PREDICTION_TARGETS:
            metrics_str += f"| {target}: {result['test_metrics'].get(f'{target}_acc', np.nan):.4f} "
        print(metrics_str)
    else:
        print("Skipped: insufficient samples")


##########################################################################################
### MTL: DIRECTION
##########################################################################################

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6537 | Val loss: 0.6413
Test Accuracies: | EBITDA: 0.6432 | Net_Income: 0.5318 | ROA: 0.4702 

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6536 | Val loss: 0.6405
Test Accuracies: | EBITDA: 0.6432 | Net_Income: 0.5236 | ROA: 0.5051 

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6381 | Val loss: 0.6335
Test Accuracies: | EBITDA: 0.6432 | Net_Income: 0.5216 | ROA: 0.5051 

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6416 | Val loss: 0.6303
Test Accuracies: | EBITDA: 0.6432 | Net_Income: 0.4784 | ROA: 0.4969 

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.6501 | Val loss: 0.6374
Test Accuracies: | EBITDA: 0.6432 | Net_In

In [ ]:
run_results_mtl_value = []

print(f"\n{'#'*90}")
print("### MTL: VALUE")
print(f"{'#'*90}")

for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
    print(
        f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
    )
    result = fit_single_experiment_mtl(
        config=config,
        max_seq_length=max_seq_length,
        task="value",
    )

    if result is not None:
        run_results_mtl_value.append(result)
        metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
        metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}\n"
        metrics_str += "Test MAE: "
        for target in PREDICTION_TARGETS:
            metrics_str += f"| {target}: {result['test_metrics'].get(f'{target}_mae', np.nan):.4f} "
        print(metrics_str)
    else:
        print("Skipped: insufficient samples")


##########################################################################################
### MTL: VALUE
##########################################################################################

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 3.3449 | Val loss: 7.5720
Test MAE: | EBITDA: 1.4797 | Net_Income: 1.5852 | ROA: 1.5902 

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 2.2420 | Val loss: 7.5752
Test MAE: | EBITDA: 1.4837 | Net_Income: 1.5730 | ROA: 1.5943 

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 3.3607 | Val loss: 7.5676
Test MAE: | EBITDA: 1.4919 | Net_Income: 1.5593 | ROA: 1.6181 

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 3.3264 | Val loss: 7.5642
Test MAE: | EBITDA: 1.4965 | Net_Income: 1.5616 | ROA: 1.6020 

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 3.3007 | Val loss: 7.5690
Test MAE: | EBITDA: 1.4876 | Net_Income: 1.5587 | ROA: 1.6393 

Config 6/3

## Single Task Learning (STL) Implementations

In [ ]:
run_results_direction = []

print(f"\n{'#'*90}")
print("### RUNNING TASK: DIRECTION")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training shallow LSTM | Task: DIRECTION | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment(
            target_name=target_name,
            config=config,
            max_seq_length=max_seq_length,
            task="direction",
        )

        if result is not None:
            run_results_direction.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train acc: {result['train_metrics']['accuracy']:.4f}"
                f" | Val acc: {result['val_metrics']['accuracy']:.4f}"
                f" | Test acc: {result['test_metrics']['accuracy']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### RUNNING TASK: DIRECTION
##########################################################################################

Training shallow LSTM | Task: DIRECTION | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4720 | Val loss: 0.4813 | Train acc: 0.6106 | Val acc: 0.6380 | Test acc: 0.6090

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4776 | Val loss: 0.4822 | Train acc: 0.6122 | Val acc: 0.7362 | Test acc: 0.6278

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4730 | Val loss: 0.4799 | Train acc: 0.6201 | Val acc: 0.6319 | Test acc: 0.5489

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4730 | Val loss: 0.4751 | Train acc: 0.6201 | Val acc: 0.7362 | Test acc: 0.5038

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.4684 | Val loss: 0.4833 | Train acc: 0.6186 | Val a

In [ ]:
run_results_value = []

print(f"\n{'#'*90}")
print("### RUNNING TASK: VALUE")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training shallow LSTM | Task: VALUE | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment(
            target_name=target_name,
            config=config,
            max_seq_length=max_seq_length,
            task="value",
        )

        if result is not None:
            run_results_value.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train MAE: {result['train_metrics']['mae']:.4f}"
                f" | Val MAE: {result['val_metrics']['mae']:.4f}"
                f" | Test MAE: {result['test_metrics']['mae']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### RUNNING TASK: VALUE
##########################################################################################

Training shallow LSTM | Task: VALUE | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.7679 | Val loss: 0.8593 | Train MAE: 1.0197 | Val MAE: 1.1488 | Test MAE: 1.2560

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.7679 | Val loss: 0.8592 | Train MAE: 1.0197 | Val MAE: 1.1486 | Test MAE: 1.2559

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.7389 | Val loss: 0.8599 | Train MAE: 0.9859 | Val MAE: 1.1468 | Test MAE: 1.2431

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.7389 | Val loss: 0.8596 | Train MAE: 0.9859 | Val MAE: 1.1463 | Test MAE: 1.2430

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.7668 | Val loss: 0.8594 | Train MAE: 1.0167 | Val MAE: 1.14

# Outlier Handling, Sequence Packing & Schedulers

This section introduces an updated dataset class, model architecture, and training loop reflecting the following improvements without overwriting the previous outputs:
1. **Handling long tails**: Implements `RobustScaler` (instead of `StandardScaler`) and extreme value clipping (Winsorization) for regression targets.
2. **Feature Selection Filter**: Adds a placeholder to easily filter inputs down to those found in the `features_selection` notebook.
3. **Optimized LSTMs (`pack_padded_sequence`)**: Pre-padding zeros can negatively impact LSTM hidden states. This is updated to use variable-length sequences properly.
4. **Learning Rate Schedulers & Early Stopping**: Uses `ReduceLROnPlateau` and monitors validation metrics (Accuracy/MAE) directly instead of BCE loss.


In [13]:
from sklearn.preprocessing import RobustScaler
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. Select specific features here based on your EDA (using a subset as placeholder)
# e.g., SELECTED_FEATURES = ["Feature1", "Feature2"]
SELECTED_FEATURES = feature_cols.copy() # Replace this list with top features from features_selection.ipynb
selected_indices = [feature_cols.index(f) for f in SELECTED_FEATURES]

# 2. Winsorize target values (reduce impact of extreme tail values)
improved_data_df = data_df.copy()
idx_value = improved_data_df["target"].isin(PREDICTION_TARGETS)
q_low = improved_data_df.loc[idx_value, "label_value"].quantile(0.01)
q_high = improved_data_df.loc[idx_value, "label_value"].quantile(0.99)
improved_data_df.loc[idx_value, "label_value"] = improved_data_df.loc[idx_value, "label_value"].clip(q_low, q_high)

train_data_imp, val_data_imp, test_data_imp = split_by_year(improved_data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

# 3. Robust Scaling
robust_scaler = RobustScaler()
train_windows_imp = np.vstack([row[:, selected_indices] for row in train_data_imp["window_data"]])
robust_scaler.fit(train_windows_imp)

class ImprovedYoYDataset(Dataset):
    def __init__(self, data_df, task, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.task = task
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        # Filter features
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler:
            window = self.scaler.transform(window)

        target = np.float32(row[f"label_{self.task}"])
        # Return actual unpadded window and length for pack_padded_sequence
        return torch.from_numpy(window), torch.tensor(target), len(window)

def collate_fn_improved(batch):
    # Sort batch by sequence length (descending) for pack_padded_sequence
    batch.sort(key=lambda x: x[2], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    lengths = torch.tensor([x[2] for x in batch])

    # Pad sequences in the batch
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, lengths

class ImprovedShallowLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        # Pack the sequence to ignore padding during LSTM computation
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, (hn, cn) = self.lstm(packed_x)
        # Use final hidden state
        hidden = self.dropout(hn[-1])
        logits = self.head(hidden).squeeze(-1)
        return logits

# Note: You can now update fit_single_experiment to use ImprovedYoYDataset, collate_fn_improved
# in the DataLoader, pass `lengths` to model(batch_x, lengths), and initialize ReduceLROnPlateau.

In [14]:
@torch.no_grad()
def collect_predictions_improved(model, loader, criterion, task="direction"):
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds = []
    all_targets = []

    for batch_x, batch_y, lengths in loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        logits = model(batch_x, lengths)
        loss = criterion(logits, batch_y)

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

        if task == "direction":
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.cpu().numpy())

        all_targets.append(batch_y.cpu().numpy())

    if total_count == 0:
        return {"loss": np.nan, "accuracy": np.nan, "roc_auc": np.nan}, np.array([]), np.array([])

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    if task == "direction":
        y_pred_binary = (y_pred >= 0.5).astype(int)
        try:
            auc = roc_auc_score(y_true.astype(int), y_pred)
        except ValueError:
            auc = np.nan
        metrics = {
            "loss": total_loss / total_count,
            "accuracy": accuracy_score(y_true.astype(int), y_pred_binary),
            "roc_auc": auc,
        }
        return metrics, y_true.astype(int), y_pred
    else:
        mae = np.mean(np.abs(y_pred - y_true))
        metrics = {"loss": total_loss / total_count, "mae": mae}
        return metrics, y_true, y_pred


def train_epoch_improved(model, loader, criterion, optimizer, task="direction"):
    model.train()
    total_loss = 0.0
    total_count = 0

    for batch_x, batch_y, lengths in loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x, lengths)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = len(batch_y)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

    return {"loss": total_loss / total_count if total_count > 0 else np.nan}


def fit_single_experiment_improved(target_name: str, config: dict, task: str):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    target_train = train_data_imp[train_data_imp["target"] == target_name].copy()
    target_val = val_data_imp[val_data_imp["target"] == target_name].copy()
    target_test = test_data_imp[test_data_imp["target"] == target_name].copy()

    if len(target_train) < 5 or len(target_val) < 2 or len(target_test) < 2:
        return None

    train_ds = ImprovedYoYDataset(target_train, task, robust_scaler, selected_indices)
    val_ds = ImprovedYoYDataset(target_val, task, robust_scaler, selected_indices)
    test_ds = ImprovedYoYDataset(target_test, task, robust_scaler, selected_indices)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn_improved)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn_improved)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn_improved)

    model = ImprovedShallowLSTM(input_size=len(selected_indices), hidden_size=config["hidden_size"]).to(DEVICE)

    if task == "direction":
        pos_weight = torch.tensor(
            (len(target_train) - target_train["label_direction"].sum()) / max(1, target_train["label_direction"].sum()),
            dtype=torch.float32, device=DEVICE
        )
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.HuberLoss(delta=1.0)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    # Track accuracy for direction, MAE for value
    mode = 'max' if task == 'direction' else 'min'
    scheduler = ReduceLROnPlateau(optimizer, mode=mode, patience=5, factor=0.5)

    best_state = None
    best_metric = -float("inf") if task == "direction" else float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        _ = train_epoch_improved(model, train_loader, criterion, optimizer, task)
        val_metrics, _, _ = collect_predictions_improved(model, val_loader, criterion, task)

        current_metric = val_metrics["accuracy"] if task == "direction" else val_metrics["mae"]
        scheduler.step(current_metric)

        improved = current_metric > best_metric if task == "direction" else current_metric < best_metric

        if improved:
            best_metric = current_metric
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_metrics, _, _ = collect_predictions_improved(model, train_loader, criterion, task)
    val_metrics, _, _ = collect_predictions_improved(model, val_loader, criterion, task)
    test_metrics, _, _ = collect_predictions_improved(model, test_loader, criterion, task)

    return {
        "target": target_name,
        "config": config,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

In [ ]:
run_results_direction_imp = []

print(f"\n{'#'*90}")
print("### IMPROVED PIPELINE: DIRECTION")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training improved shallow LSTM | Task: DIRECTION | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment_improved(
            target_name=target_name,
            config=config,
            task="direction",
        )

        if result is not None:
            run_results_direction_imp.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train acc: {result['train_metrics']['accuracy']:.4f}"
                f" | Val acc: {result['val_metrics']['accuracy']:.4f}"
                f" | Test acc: {result['test_metrics']['accuracy']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### IMPROVED PIPELINE: DIRECTION
##########################################################################################

Training improved shallow LSTM | Task: DIRECTION | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4690 | Val loss: 0.4818 | Train acc: 0.5998 | Val acc: 0.6780 | Test acc: 0.5290

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.4692 | Val loss: 0.4850 | Train acc: 0.6007 | Val acc: 0.6814 | Test acc: 0.5436

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4775 | Val loss: 0.4907 | Train acc: 0.6078 | Val acc: 0.6712 | Test acc: 0.5581

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.4720 | Val loss: 0.4825 | Train acc: 0.6069 | Val acc: 0.6746 | Test acc: 0.5394

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.4734 | Val loss: 0.4919 | Train acc: 

In [ ]:
run_results_value_imp = []

print(f"\n{'#'*90}")
print("### IMPROVED PIPELINE: VALUE")
print(f"{'#'*90}")

for target_name in PREDICTION_TARGETS:
    print(f"\n{'='*90}")
    print(f"Training improved shallow LSTM | Task: VALUE | Target: {target_name}")
    print(f"{'='*90}")

    for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
        print(
            f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
        )
        result = fit_single_experiment_improved(
            target_name=target_name,
            config=config,
            task="value",
        )

        if result is not None:
            run_results_value_imp.append(result)
            metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
            metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}"
            metrics_str += (
                f" | Train MAE: {result['train_metrics']['mae']:.4f}"
                f" | Val MAE: {result['val_metrics']['mae']:.4f}"
                f" | Test MAE: {result['test_metrics']['mae']:.4f}"
            )
            print(metrics_str)
        else:
            print("Skipped: insufficient samples")


##########################################################################################
### IMPROVED PIPELINE: VALUE
##########################################################################################

Training improved shallow LSTM | Task: VALUE | Target: EBITDA

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6872 | Val loss: 0.8076 | Train MAE: 0.9393 | Val MAE: 1.1012 | Test MAE: 0.8205

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6873 | Val loss: 0.8070 | Train MAE: 0.9394 | Val MAE: 1.1001 | Test MAE: 0.8198

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6880 | Val loss: 0.8057 | Train MAE: 0.9416 | Val MAE: 1.1018 | Test MAE: 0.8296

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6881 | Val loss: 0.8056 | Train MAE: 0.9415 | Val MAE: 1.1013 | Test MAE: 0.8294

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.6893 | Val loss: 0.8070 | Train MAE: 0.9415 |

## Improved Multi-Task Learning (MTL) Pipelines

This section applies the same improvements (Winsorization, RobustScaler, sequence packing, and ReduceLROnPlateau scheduling) to the Multi-Task Learning pipelines. This allows predicting all targets simultaneously using the optimized techniques.

In [15]:
mtl_train_data_imp = extract_mtl_df(train_data_imp)
mtl_val_data_imp = extract_mtl_df(val_data_imp)
mtl_test_data_imp = extract_mtl_df(test_data_imp)

class ImprovedYoYDatasetMTL(Dataset):
    def __init__(self, data_df: pd.DataFrame, task: str, scaler, feature_indices: list[int]):
        self.data_df = data_df.reset_index(drop=True)
        self.task = task
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self) -> int:
        return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()

        if self.scaler is not None:
            window = self.scaler.transform(window)

        target = row[f"label_{self.task}"]

        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)

        # Return unpadded window, target, mask, and sequence length
        return (
            torch.from_numpy(window),
            torch.tensor(target_clean, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.bool),
            len(window)
        )

def collate_fn_improved_mtl(batch):
    batch.sort(key=lambda x: x[3], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    masks = torch.stack([x[2] for x in batch])
    lengths = torch.tensor([x[3] for x in batch])

    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, lengths

class ImprovedShallowLSTM_MTL(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = len(PREDICTION_TARGETS), task: str = "direction"):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, num_targets)
        self.task = task

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        packed_out, (hn, cn) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        logits = self.head(hidden)
        return logits

def train_epoch_improved_mtl(model, loader, optimizer, task="direction"):
    model.train()
    total_loss = 0.0
    total_count = 0

    if task == "direction":
        criterion = nn.BCEWithLogitsLoss(reduction='none')
    else:
        criterion = nn.HuberLoss(delta=1.0, reduction='none')

    for batch_x, batch_y, batch_mask, lengths in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        batch_mask = batch_mask.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_x, lengths)

        loss_matrix = criterion(logits, batch_y)
        masked_loss = loss_matrix[batch_mask]

        if masked_loss.numel() > 0:
            loss = masked_loss.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += float(loss.item()) * batch_x.size(0)
            total_count += batch_x.size(0)

    return {"loss": total_loss / max(1, total_count)}

@torch.no_grad()
def collect_predictions_improved_mtl(model, loader, task="direction"):
    model.eval()
    total_loss = 0.0
    total_count = 0
    all_preds = []
    all_targets = []
    all_masks = []

    if task == "direction":
        criterion = nn.BCEWithLogitsLoss(reduction='none')
    else:
        criterion = nn.HuberLoss(delta=1.0, reduction='none')

    for batch_x, batch_y, batch_mask, lengths in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        batch_mask = batch_mask.to(DEVICE)

        logits = model(batch_x, lengths)
        loss_matrix = criterion(logits, batch_y)
        masked_loss = loss_matrix[batch_mask]

        if masked_loss.numel() > 0:
            total_loss += float(masked_loss.mean().item()) * batch_x.size(0)
            total_count += batch_x.size(0)

        if task == "direction":
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(probs)
        else:
            all_preds.append(logits.cpu().numpy())

        all_targets.append(batch_y.cpu().numpy())
        all_masks.append(batch_mask.cpu().numpy())

    if len(all_preds) == 0:
        return {"loss": np.nan}, np.array([]), np.array([]), np.array([])

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)
    mask = np.vstack(all_masks)

    metrics = {"loss": total_loss / max(1, total_count)}

    # Calculate overall avg metric for scheduler
    total_metric_sum = 0.0
    valid_targets = 0

    for i, target_name in enumerate(PREDICTION_TARGETS):
        m = mask[:, i]
        if m.sum() == 0:
            continue

        target_true = y_true[m, i]
        target_pred = y_pred[m, i]

        if task == "direction":
            target_pred_binary = (target_pred >= 0.5).astype(int)
            acc = accuracy_score(target_true.astype(int), target_pred_binary)
            metrics[f"{target_name}_acc"] = acc
            total_metric_sum += acc
            valid_targets += 1
        else:
            mae = np.mean(np.abs(target_pred - target_true))
            metrics[f"{target_name}_mae"] = mae
            total_metric_sum += mae
            valid_targets += 1

    if valid_targets > 0:
        metrics["overall_metric"] = total_metric_sum / valid_targets
    else:
        metrics["overall_metric"] = np.nan

    return metrics, y_true, y_pred, mask

def fit_single_experiment_improved_mtl(config: dict, task: str):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, task, robust_scaler, selected_indices)
    val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, task, robust_scaler, selected_indices)
    test_ds = ImprovedYoYDatasetMTL(mtl_test_data_imp, task, robust_scaler, selected_indices)

    if len(mtl_train_data_imp) < 5 or len(mtl_val_data_imp) < 2 or len(mtl_test_data_imp) < 2:
        return None

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, collate_fn=collate_fn_improved_mtl)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, collate_fn=collate_fn_improved_mtl)

    model = ImprovedShallowLSTM_MTL(input_size=len(selected_indices), hidden_size=config["hidden_size"], task=task).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    mode = 'max' if task == 'direction' else 'min'
    scheduler = ReduceLROnPlateau(optimizer, mode=mode, patience=5, factor=0.5)

    best_state = None
    best_metric = -float("inf") if task == "direction" else float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        _ = train_epoch_improved_mtl(model, train_loader, optimizer, task)
        val_metrics, _, _, _ = collect_predictions_improved_mtl(model, val_loader, task)

        current_metric = val_metrics.get("overall_metric", np.nan)
        if not np.isnan(current_metric):
            scheduler.step(current_metric)
            improved = current_metric > best_metric if task == "direction" else current_metric < best_metric

            if improved:
                best_metric = current_metric
                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_metrics, _, _, _ = collect_predictions_improved_mtl(model, train_loader, task)
    val_metrics, _, _, _ = collect_predictions_improved_mtl(model, val_loader, task)
    test_metrics, test_true, test_pred, test_mask = collect_predictions_improved_mtl(model, test_loader, task)

    return {
        "config": config,
        "model": model,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }

In [16]:
run_results_mtl_direction_imp = []

print(f"\n{'#'*90}")
print("### IMPROVED MTL: DIRECTION")
print(f"{'#'*90}")

for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
    print(
        f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
    )
    result = fit_single_experiment_improved_mtl(
        config=config,
        task="direction",
    )

    if result is not None:
        run_results_mtl_direction_imp.append(result)
        metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
        metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}\n"
        metrics_str += "Test Accuracies: "
        for target in PREDICTION_TARGETS:
            metrics_str += f"| {target}: {result['test_metrics'].get(f'{target}_acc', np.nan):.4f} "
        print(metrics_str)
    else:
        print("Skipped: insufficient samples")


##########################################################################################
### IMPROVED MTL: DIRECTION
##########################################################################################

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6501 | Val loss: 0.6514
Test Accuracies: | EBITDA: 0.6390 | Net_Income: 0.5606 | ROA: 0.5400 

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 0.6504 | Val loss: 0.6513
Test Accuracies: | EBITDA: 0.6390 | Net_Income: 0.5585 | ROA: 0.5359 

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6475 | Val loss: 0.6472
Test Accuracies: | EBITDA: 0.6411 | Net_Income: 0.5565 | ROA: 0.5277 

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 0.6447 | Val loss: 0.6458
Test Accuracies: | EBITDA: 0.6411 | Net_Income: 0.5544 | ROA: 0.5380 

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 0.6423 | Val loss: 0.6469
Test Accuracies: | EBITDA: 0.6390

In [17]:
run_results_mtl_value_imp = []

print(f"\n{'#'*90}")
print("### IMPROVED MTL: VALUE")
print(f"{'#'*90}")

for config_idx, config in enumerate(DEFAULT_EXPERIMENTS):
    print(
        f"\nConfig {config_idx + 1}/{len(DEFAULT_EXPERIMENTS)} | hidden_size={config['hidden_size']} | lr={config['lr']} | batch_size={config['batch_size']}"
    )
    result = fit_single_experiment_improved_mtl(
        config=config,
        task="value",
    )

    if result is not None:
        run_results_mtl_value_imp.append(result)
        metrics_str = f"Train loss: {result['train_metrics']['loss']:.4f}"
        metrics_str += f" | Val loss: {result['val_metrics']['loss']:.4f}\n"
        metrics_str += "Test MAE: "
        for target in PREDICTION_TARGETS:
            metrics_str += f"| {target}: {result['test_metrics'].get(f'{target}_mae', np.nan):.4f} "
        print(metrics_str)
    else:
        print("Skipped: insufficient samples")


##########################################################################################
### IMPROVED MTL: VALUE
##########################################################################################

Config 1/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 1.0098 | Val loss: 1.1853
Test MAE: | EBITDA: 0.8288 | Net_Income: 1.3011 | ROA: 1.2886 

Config 2/36 | hidden_size=32 | lr=0.001 | batch_size=16
Train loss: 1.0103 | Val loss: 1.1850
Test MAE: | EBITDA: 0.8276 | Net_Income: 1.3004 | ROA: 1.2858 

Config 3/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 1.0069 | Val loss: 1.1827
Test MAE: | EBITDA: 0.8420 | Net_Income: 1.3033 | ROA: 1.2931 

Config 4/36 | hidden_size=32 | lr=0.001 | batch_size=32
Train loss: 1.0054 | Val loss: 1.1809
Test MAE: | EBITDA: 0.8434 | Net_Income: 1.3008 | ROA: 1.2844 

Config 5/36 | hidden_size=32 | lr=0.0003 | batch_size=16
Train loss: 1.0070 | Val loss: 1.1850
Test MAE: | EBITDA: 0.8396 | Net_Income: 1.3066 | ROA: 1.2933 

C